# Design Robustness

`permanova.ipynb` fits the factorial model on all 80 rows. That treats the five repetitions of a cell
as independent observations, which they are not: they are technical replicates recorded back to back
under one configuration. This notebook works through the consequences and reports the checks that a
reader is entitled to ask for.

Four questions are addressed:

1. How large is the comb material effect in interpretable units, and is it bounded?
2. Does the natural-to-bent ratio survive correction for the pitch difference it carries?
3. Does it survive removing either experimental block?
4. What is the smallest p-value the design can produce once the replicate structure is respected?

## Blocks

The block is not stored in `descriptors_v2.csv`, but it is recoverable. Block A recorded each
harmonica in its factory configuration, C with the wooden comb and B flat with the ABS comb; block B
recorded the swapped pairings. The block is therefore exactly the harmonica by material interaction.

Two consequences follow. The comb main effect is balanced across blocks, twenty recordings of each
comb in each block, so it is estimable. The harmonica by material interaction, on the other hand, is
perfectly aliased with the block and cannot be separated from it.

In [ ]:
import csv
import itertools
import numpy as np

rng = np.random.default_rng(20260601)

rows = list(csv.DictReader(open('../data/descriptors_v2.csv')))
for r in rows:
    r['centroid'] = float(r['centroid'])
    r['f0'] = float(r['f0'])
    r['block'] = 'A' if (r['harp'], r['material']) in (('C', 'wood'), ('Bb', 'abs')) else 'B'

CELLS = [(n, m) for n in ('E4', 'F4', 'G4', 'A4') for m in ('natural', 'bend')]
select = lambda **kw: [r for r in rows if all(r[k] == v for k, v in kw.items())]

from collections import Counter
print(Counter((r['material'], r['block']) for r in rows))

## 1. Comb material: effect size and equivalence bound

A p-value of 0.31 does not establish that comb material has no effect; it only fails to establish
that it has one. The question a maker or a player actually asks is how large the effect could be, and
that calls for an interval rather than a test.

The estimand is the wood minus ABS difference of cell medians, expressed as a percentage of the cell
level, averaged over the eight note by method cells. The interval is a BCa bootstrap resampling the
five repetitions within each cell and material, which keeps the design fixed and respects the
replicate structure.

In [ ]:
def comb_effect(sample):
    diffs = []
    for note, method in CELLS:
        wood = [r['centroid'] for r in sample
                if r['note'] == note and r['method'] == method and r['material'] == 'wood']
        abs_ = [r['centroid'] for r in sample
                if r['note'] == note and r['method'] == method and r['material'] == 'abs']
        if not wood or not abs_:
            return np.nan
        mw, ma = np.median(wood), np.median(abs_)
        diffs.append((mw - ma) / ((mw + ma) / 2) * 100)
    return float(np.mean(diffs))


def resample_within_cells():
    out = []
    for note, method in CELLS:
        for material in ('wood', 'abs'):
            g = select(note=note, method=method, material=material)
            out += [g[i] for i in rng.integers(0, len(g), len(g))]
    return out


def normal_quantile(p):
    a = [-3.969683028665376e+01, 2.209460984245205e+02, -2.759285104469687e+02,
         1.383577518672690e+02, -3.066479806614716e+01, 2.506628277459239e+00]
    b = [-5.447609879822406e+01, 1.615858368580409e+02, -1.556989798598866e+02,
         6.680131188771972e+01, -1.328068155288572e+01]
    c = [-7.784894002430293e-03, -3.223964580411365e-01, -2.400758277161838e+00,
         -2.549732539343734e+00, 4.374664141464968e+00, 2.938163982698783e+00]
    d = [7.784695709041462e-03, 3.224671290700398e-01, 2.445134137142996e+00,
         3.754408661907416e+00]
    if p < 0.02425:
        q = np.sqrt(-2 * np.log(p))
        return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
    if p > 1 - 0.02425:
        q = np.sqrt(-2 * np.log(1 - p))
        return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
    q, r = p - 0.5, (p - 0.5) ** 2
    return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q / (((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)


def normal_cdf(z):
    from math import erf
    return 0.5 * (1 + erf(z / np.sqrt(2)))


def bca_interval(theta, boots, jackknife, level=0.90):
    boots = np.asarray(boots)
    z0 = normal_quantile(max((boots < theta).mean(), 1e-6))
    jm = jackknife.mean()
    denom = 6 * (((jm - jackknife) ** 2).sum() ** 1.5)
    a = ((jm - jackknife) ** 3).sum() / denom if denom else 0.0
    tail = (1 - level) / 2
    out = []
    for alpha in (tail, 1 - tail):
        z = normal_quantile(alpha)
        out.append(normal_cdf(z0 + (z0 + z) / (1 - a * (z0 + z))))
    return float(np.quantile(boots, out[0])), float(np.quantile(boots, out[1]))


theta = comb_effect(rows)
boots = np.array([comb_effect(resample_within_cells()) for _ in range(4000)])
boots = boots[~np.isnan(boots)]
jack = np.array([v for v in (comb_effect(rows[:i] + rows[i + 1:]) for i in range(len(rows)))
                 if not np.isnan(v)])
low, high = bca_interval(theta, boots, jack)

print(f'wood minus ABS: {theta:+.2f}% of cell level')
print(f'BCa 90% interval: [{low:+.2f}%, {high:+.2f}%]')
for margin in (5, 10):
    print(f'  equivalence within +-{margin}%: {"established" if low > -margin and high < margin else "not established"}')

print('\nby cell:')
for note, method in CELLS:
    w = np.median([r['centroid'] for r in select(note=note, method=method, material='wood')])
    a = np.median([r['centroid'] for r in select(note=note, method=method, material='abs')])
    print(f'  {note} {method:8} {(w - a) / ((w + a) / 2) * 100:+7.2f}%')

## 2. Pitch correction of the within-plate contrast

On the B flat plates, hole 2 draw yields F4 naturally and E4 as a bend, and hole 3 draw yields A4
naturally and G4 as a bend. Those two pairs share reed plate, hole and airflow direction, so they
isolate the technique better than any comparison across instruments. They do, however, carry a pitch
difference of one and two semitones, always with the natural note higher.

Under the least favourable assumption, that the centroid scales with $f_0$, the ratio is divided by
the measured $f_0$ ratio.

In [ ]:
for hole, natural_note, bent_note in (('hole 2', 'F4', 'E4'), ('hole 3', 'A4', 'G4')):
    nat = select(harp='Bb', note=natural_note, method='natural')
    ben = select(harp='Bb', note=bent_note, method='bend')
    ratio = np.median([r['centroid'] for r in nat]) / np.median([r['centroid'] for r in ben])
    f0_ratio = np.median([r['f0'] for r in nat]) / np.median([r['f0'] for r in ben])
    print(f'{hole}: {ratio:.2f}:1 raw, {ratio / f0_ratio:.2f}:1 after pitch correction '
          f'(f0 ratio {f0_ratio:.4f})')

## 3. Leave-one-block-out

The contrast should not depend on either half of the session. Removing a block halves the sample, so
the interest is in the direction and the order of magnitude, not in the third decimal.

In [ ]:
for hole, natural_note, bent_note in (('hole 2', 'F4', 'E4'), ('hole 3', 'A4', 'G4')):
    for dropped in (None, 'A', 'B'):
        nat = [r for r in select(harp='Bb', note=natural_note, method='natural') if r['block'] != dropped]
        ben = [r for r in select(harp='Bb', note=bent_note, method='bend') if r['block'] != dropped]
        ratio = np.median([r['centroid'] for r in nat]) / np.median([r['centroid'] for r in ben])
        label = 'full' if dropped is None else f'without block {dropped}'
        print(f'{hole} {label:18} n={len(nat)}+{len(ben)}  ratio {ratio:.2f}:1')

## 4. Attainable p-values

Permutation tests cannot return a p-value smaller than the reciprocal of the number of distinct
labellings. Once the five repetitions are held together as one experimental unit, the number of
labellings drops sharply, and with it the resolution of the test.

The consequence is that a p-value is the wrong summary for the within-plate contrast: under a
permutation that respects the blocking, no effect of any size can reach the conventional threshold.
Interval estimation, as in section 1, does not have that ceiling.

In [ ]:
schemes = [
    ('8 units, sign flips within 4 comb-by-hole blocks', 1 / 2 ** 4),
    ('8 units, free 4 versus 4 permutation', 1 / len(list(itertools.combinations(range(8), 4)))),
    ('16 cells, sign flips within 8 blocks', 1 / 2 ** 8),
    ('80 rows, free permutation (as currently fitted)', 1 / 9999),
]
for label, floor in schemes:
    print(f'{label:52} smallest attainable p = {floor:.4f}')